# Transformer 架构

![图片描述](images/Self-Attention.png)

![图片描述](images/Transformer%20架构.png)

## 神经网络的三种核心架构
* 前馈神经网络(Feedforward Neural Nerwork,FNN):从输入层单向流动到输出层，无循环。例如多层感知机、每层的神经元都和剩下两层的每个神经元完全连接
* 卷积神经网络(Convolutional Neural Network,CNN):训练参数小于全连接神经网络的卷积层，可以用来进行特征提取和学习。
* 循环神经网络(Recurrent Neural Network,RNN):能使用历史信息作为输入，能够自循环的网络
* * RNN可以专门用于处理序列以及时序的数据，但是他也有缺点：
* * * 1、因为序列需要依次输入，没办法并行计算，时间成本很高；
* * * 2、在RNN中距离越远的输入之间的关系就越难以被捕捉，并且RNN需要将整个序列读入内存以此计算，所以在一定程度上限制了序列的长度。


## 注意力核心代码实现：

In [1]:
import torch
import math

from openai.types import Embedding
from sspicon import SECPKG_FLAG_TOKEN_ONLY


# 注意力计算函数：
def attention(q,k,v,dropout = None):
    """
    args:
    query: 查询值矩阵
    key: 键值矩阵
    value: 真值矩阵
    """
    # 获取k向量的维度
    d_k = q.size(-1)
    # 计算q与k的内积并除以d_k   transpose(-2,-1)-将最后两个维度互换
    scores = torch.matmul(q,k.transpose(-2,-1)) / math.sqrt(d_k)

    # 进行指数函数计算，softmax 得到注意力分数
    p_attn = scores.softmax(dim=-1)
    # 如果dropout不为空，就对注意力权重随机失活(正则化)，防止过拟合
    if dropout is not None:
        p_attn = dropout(p_attn)

    # 返回 注意力权重跟v相乘再求和的结果
    return torch.matmul(p_attn,v),p_attn

## 自注意力机制与注意力机制的区别:
* 编码器中的自注意力机制的Q、K、V是由同一个输入向量通过不同的参数矩阵得到的。这样就可以找出一段序列中的每个词元跟其他所有词元的相关性。

## 掩码注意力：
* 创建一个下三角为1，上三角为-inf的矩阵，在进行注意力计算时，会将注意力分数与该掩码求和，在进行Softmax操作。

In [2]:
# 创建掩码矩阵
max_seq_len = 5 # 矩阵大小
# 创建全‘-inf’的矩阵
mask = torch.full((1, max_seq_len, max_seq_len), float("-inf"))
# triu 函数的功能是创建一个上三角矩阵
mask = torch.triu(mask, diagonal=1)
mask
# 再将注意力分数scores跟掩码矩阵相加

tensor([[[0., -inf, -inf, -inf, -inf],
         [0., 0., -inf, -inf, -inf],
         [0., 0., 0., -inf, -inf],
         [0., 0., 0., 0., -inf],
         [0., 0., 0., 0., 0.]]])

## 多头注意力：
* 就是要考虑语义多个因素


In [3]:
from dataclasses import dataclass
import torch.nn as nn
import torch
import torch.nn.functional as F

@dataclass
class ModelArgs:
    n_layers: int = 2  # 层数，模型深度
    vocab_size: int = 16 # 词表大小，总共认识多少个不同的token
    max_seq_len: int = 16 # 最大序列长度 意味着一次能处理最多多少个token
    dim: int = 8 # 隐藏层的维度，每个词向量的长度
    n_heads: int = 4  # 头的数量，dim必须能被头整除，每个头处理的注意力向量维度为8/4=2
    dropout_p: float = 0.1 # 正则化 失活百分之多少神经元
    use_attn_mask: bool = True # 是否使用注意力掩码
    weight_tying: bool = True # 权重共享，输入层跟输出层公用一组权重
    checkpoint_activations: bool = False


'''多头自注意力计算模块'''
class MultiHeadAttention(nn.Module):

    def __init__(self, args: ModelArgs, is_causal=False):
        # 构造函数
        # args: 配置对象
        super().__init__()
        # 隐藏层维度必须是头数的整数倍，因为后面我们会将输入拆成头数个矩阵
        # assert- 如果不能整除就抛出异常
        assert args.dim % args.n_heads == 0
        # 每个头的维度，等于模型维度除以头的总数。
        self.head_dim = args.dim // args.n_heads # 8/4=2
        self.n_heads = args.n_heads # 4

        # Wq, Wk, Wv 参数矩阵，每个参数矩阵为 n_embd x dim
        # 这里通过三个组合矩阵来代替了n个参数矩阵的组合，其逻辑在于矩阵内积再拼接其实等同于拼接矩阵再内积，
        # 不理解的读者可以自行模拟一下，每一个线性层其实相当于n个参数矩阵的拼接
        self.wq = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        # 输出权重矩阵，维度为 dim x dim（head_dim = dim / n_heads）
        self.wo = nn.Linear(self.n_heads * self.head_dim, args.dim, bias=False)
        # 注意力的 dropout
        self.attn_dropout = nn.Dropout(args.dropout_p)
        # 残差连接的 dropout
        self.resid_dropout = nn.Dropout(args.dropout_p)
        self.is_causal = is_causal

        # 创建一个上三角矩阵，用于遮蔽未来信息
        # 注意，因为是多头注意力，Mask 矩阵比之前我们定义的多一个维度
        if is_causal:
            mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            # 注册为模型的缓冲区
            self.register_buffer("mask", mask)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):

        # 获取批次大小和序列长度，[batch_size, seq_len, dim]
        bsz, seqlen, _ = q.shape

        # 计算查询（Q）、键（K）、值（V）,输入通过参数矩阵层，维度为 (B, T, n_embed) x (n_embed, dim) -> (B, T, dim)
        xq, xk, xv = self.wq(q), self.wk(k), self.wv(v)

        # 将 Q、K、V 拆分成多头，维度为 (B, T, n_head, dim // n_head)，然后交换维度，变成 (B, n_head, T, dim // n_head)
        # 因为在注意力计算中我们是取了后两个维度参与计算
        # 为什么要先按B*T*n_head*C//n_head展开再互换1、2维度而不是直接按注意力输入展开，是因为view的展开方式是直接把输入全部排开，
        # 然后按要求构造，可以发现只有上述操作能够实现我们将每个头对应部分取出来的目标
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_heads, self.head_dim)
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)

        # 注意力计算
        # 计算 QK^T / sqrt(d_k)，维度为 (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        # 掩码自注意力必须有注意力掩码
        if self.is_causal:
            assert hasattr(self, 'mask')
            # 这里截取到序列长度，因为有些序列可能比 max_seq_len 短
            scores = scores + self.mask[:, :, :seqlen, :seqlen]
        # 计算 softmax，维度为 (B, nh, T, T)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        # 做 Dropout
        scores = self.attn_dropout(scores)
        # V * Score，维度为(B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        output = torch.matmul(scores, xv)

        # 恢复时间维度并合并头。
        # 将多头的结果拼接起来, 先交换维度为 (B, T, n_head, dim // n_head)，再拼接成 (B, T, n_head * dim // n_head)
        # contiguous 函数用于重新开辟一块新内存存储，因为Pytorch设置先transpose再view会报错，
        # 因为view直接基于底层存储得到，然而transpose并不会改变底层存储，因此需要额外存储
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)

        # 最终投影回残差流。
        output = self.wo(output)
        output = self.resid_dropout(output)
        return output

## 前馈神经网络实现：

In [4]:
class MLP(nn.Module):
    def __init__(self,dim:int , hidden_dim:int,dropout:float):
        super().__init__()
        # 定义第一层线性变换，从输入到隐藏维度
        self.w1 = nn.Linear(dim , hidden_dim , bias=False)
        # 激活函数
        # 定义第二层线性变换，从隐藏到输入
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        # dropout层
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        # 先经过一层线性变换和ReLu激活函数，再进行第二层先线性变换跟dropout
        return self.dropout(self.w2(F.relu(self.w1(x))))


## 层归一化(Layer Norm)
* 不同层的取值范围或分布能够比较一致
* 统计每一层所有样本的均值和方差,从而使每个样本的分布达到稳定。

In [5]:
class LayerNorm(nn.Module):
    ''' Layer Norm 层'''
    def __init__(self, features, eps=1e-6):
        super().__init__()
        # 线性矩阵做映射
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        # 在统计每个样本所有维度的值，求均值和方差
        mean = x.mean(-1, keepdim=True) # mean: [bsz, max_len, 1]
        std = x.std(-1, keepdim=True) # std: [bsz, max_len, 1]
        # 注意这里也在最后一个维度发生了广播
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## 残差连接
* 输入的不仅是上一层的输出还包括上一层的输入，当层数较深时，能避免模型的退化
* 残差链接允许最底层数据直接传到最高层。

## 完整的Encoder层实现

In [6]:
class EncoderLayer(nn.Module):
    '''Encoder层'''
    def __init__(self, args):
        super().__init__()
        # 一个 Layer 中有两个 LayerNorm，分别在 Attention 之前和 MLP 之前
        self.attention_norm = LayerNorm(args.dim)
        # Encoder 不需要掩码，传入 is_causal=False
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.fnn_norm = LayerNorm(args.dim)
        self.feed_forward = MLP(args.dim, args.dim, args.dropout_p)

    def forward(self, x):
        # Layer Norm
        norm_x = self.attention_norm(x)
        # 自注意力
        h = x + self.attention.forward(norm_x, norm_x, norm_x)
        # 经过前馈神经网络
        out = h + self.feed_forward.forward(self.fnn_norm(h))
        return out

class Encoder(nn.Module):
    '''Encoder 块'''
    def __init__(self, args):
        super(Encoder, self).__init__()
        # 一个 Encoder 由 N 个 Encoder Layer 组成
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layers)])
        self.norm = LayerNorm(args.dim)

    def forward(self, x):
        "分别通过 N 层 Encoder Layer"
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

## Decoder 块的实现
* 多了个掩码注意力

In [7]:
# ... existing code ...
class DecoderLayer(nn.Module):
    '''解码层'''

    def __init__(self, args):
        super().__init__()
        # 统一使用 args.dim
        self.attention_norm_1 = LayerNorm(args.dim)
        # Decoder 的第一个部分是 Mask Attention，传入 is_causal=True
        self.mask_attention = MultiHeadAttention(args, is_causal=True)
        self.attention_norm_2 = LayerNorm(args.dim)
        # Decoder 的第二个部分是 类似于 Encoder 的 Attention，传入 is_causal=False
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.ffn_norm = LayerNorm(args.dim)
        # 统一使用 args.dropout_p
        self.feed_forward = MLP(args.dim, args.dim * 4, args.dropout_p)

    def forward(self, x, enc_out):
        # Layer Norm
        norm_x = self.attention_norm_1(x)
        # 掩码自注意力
        x = x + self.mask_attention.forward(norm_x, norm_x, norm_x)
        # 多头注意力
        norm_x = self.attention_norm_2(x)
        h = x + self.attention.forward(norm_x, enc_out, enc_out)
        # 经过前馈神经网络
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        return out


class Decoder(nn.Module):
    '''解码器'''

    def __init__(self, args):
        super(Decoder, self).__init__()
        # 修正：使用 n_layers (复数) 和 dim
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layers)])
        self.norm = LayerNorm(args.dim)

    def forward(self, x, enc_out):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, enc_out)
        return self.norm(x)


## Embedding层与位置编码



### Embedding

* Embedding层就是词嵌入层，就是将自然语言通过分词器tokenizer切分成token并转化成一个固定的index。就是一个存储固定长度的切入向量查找表。
* * Embedding层 输入的是一个形状为（batch_size,seq_len ,1）的矩阵，第一个维度是一次批处理的数量，第二个维度是自然语言序列的长度，第三个维度是经过分词器tokenizer传唤出来的index值。
* * Embedding层  内部是一个可训练的二维权重参数矩阵(Vocab_size,embedding_dim)，对输入的值进行拼接然后输出。（batch_size，seq_len，embedding_dim）

### 位置编码
* 因为注意力机制可以并行运算，所以位置编码格外重要。
* 就是对序列中的roken的相对位置进行编码(在Transformer中使用的正余弦进行位置编码)，再将位置编码加入词向量编码中。

In [8]:
import numpy as np
import matplotlib.pyplot as plt
def PositionEncoding(seq_len, d_model, n=10000):
    # 生成全0矩阵
    P = np.zeros((seq_len, d_model))
    for k in range(seq_len):
        for i in np.arange(int(d_model/2)):
            denominator = np.power(n, 2*i/d_model)
            P[k, 2*i] = np.sin(k/denominator)
            P[k, 2*i+1] = np.cos(k/denominator)
    return P

P = PositionEncoding(seq_len=4, d_model=4, n=100)
print(P)

[[ 0.          1.          0.          1.        ]
 [ 0.84147098  0.54030231  0.09983342  0.99500417]
 [ 0.90929743 -0.41614684  0.19866933  0.98006658]
 [ 0.14112001 -0.9899925   0.29552021  0.95533649]]


In [9]:
# ... existing code ...
class PositionalEncoding(nn.Module):
    '''位置编码模块'''

    def __init__(self, args):
        super(PositionalEncoding, self).__init__()
        # Dropout 层
        # self.dropout = nn.Dropout(p=args.dropout_p)

        # max_seq_len 是序列的最大长度
        pe = torch.zeros(args.max_seq_len, args.dim)
        position = torch.arange(0, args.max_seq_len).unsqueeze(1)
        # 计算 theta
        div_term = torch.exp(
            torch.arange(0, args.dim, 2) * -(math.log(10000.0) / args.dim)
        )
        # 分别计算 sin、cos 结果
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 将位置编码加到 Embedding 结果上
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return x
# ... existing code ...


## 完整的Transformer实现


In [10]:
# ... existing code ...
class Transformer(nn.Module):
    '''整体模型'''

    def __init__(self, args):
        super().__init__()
        # 必须输入词表大小和 max_seq_len
        assert args.vocab_size is not None
        assert args.max_seq_len is not None
        self.args = args
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(args.vocab_size, args.dim),
            wpe=PositionalEncoding(args),
            drop=nn.Dropout(args.dropout_p),
            encoder=Encoder(args),
            decoder=Decoder(args),
        ))
        # 最后的线性层，输入是 dim，输出是词表大小
        self.lm_head = nn.Linear(args.dim, args.vocab_size, bias=False)

        # 初始化所有的权重
        self.apply(self._init_weights)

        # 查看所有参数的数量
        print("number of parameters: %.2fM" % (self.get_num_params() / 1e6,))

    '''统计所有参数的数量'''

    def get_num_params(self, non_embedding=False):
        # non_embedding: 是否统计 embedding 的参数
        n_params = sum(p.numel() for p in self.parameters())
        # 如果不统计 embedding 的参数，就减去
        if non_embedding:
            n_params -= self.transformer.wte.weight.numel()
        return n_params

    '''初始化权重'''

    def _init_weights(self, module):
        # 线性层和 Embedding 层初始化为正则分布
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    '''前向计算函数'''

    def forward(self, idx, targets=None):
        # 输入为 idx，维度为 (batch size, sequence length)；targets 为目标序列，用于计算 loss
        device = idx.device
        b, t = idx.size()
        assert t <= self.args.max_seq_len, f"不能计算该序列，该序列长度为 {t}, 最大序列长度只有 {self.args.max_seq_len}"

        # 通过 self.transformer
        tok_emb = self.transformer.wte(idx)
        # 然后通过位置编码
        pos_emb = self.transformer.wpe(tok_emb)
        # 再进行 Dropout
        x = self.transformer.drop(pos_emb)

        # 然后通过 Encoder
        enc_out = self.transformer.encoder(x)
        # 再通过 Decoder
        x = self.transformer.decoder(x, enc_out)

        if targets is not None:
            # 训练阶段，如果我们给了 targets，就计算 loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # 推理阶段，我们只需要 logits，loss 为 None
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

# 1. 实例化配置对象
# ... existing code ...


In [11]:
# 1. 实例化配置对象
args = ModelArgs(
    n_layers=2,
    vocab_size=100,      # 假设词表大小为100
    max_seq_len=20,      # 假设最大长度为20
    dim=32,              # 隐藏层维度设为32
    n_heads=4,           # 4个头
    dropout_p=0.1
)

# 2. 实例化模型
# 注意：请确保你的 Notebook 里已经定义了 PositionalEncoding 和 Decoder 类
# 如果还没定义 Decoder，可以先只测试 Encoder: enc = Encoder(args)
try:
    model = Transformer(args)
    print("✅ 模型初始化成功！")
except Exception as e:
    print(f"❌ 模型初始化失败: {e}")
    # 如果这里报错说 Decoder 没定义，请先补充 Decoder 代码

number of parameters: 0.05M
✅ 模型初始化成功！


# 预训练的一些语言模型

## Encoder-Only PLM (只有编码器的预训练语言模型)

### BERT
* 分类模型，输入文本序列，输出label标签
* 正式确立了预训练+微调的两阶段思想
* * 预训练：在海量无监督预料上进行预训练来获得通用的文本理解与生成能力
* * 微调：在对应的下游任务上进行微调
* MLM (掩码语言模型)
* * 具体进行 MLM 训练时,会随机选择训练语料中15%的token(在15%的token中**80% 遮蔽 + 10% 随机替换 + 10% 不变**)用于遮蔽

#### BERT的基本运行流程
* 输入文本序列会先通过tokenizer(分词器)转化成input_ids
* 然后进入Embedding(词嵌入)层转化成特定位的的hidden_states
* 在进入Encoder(解码器)Layer，
* * 进入 Attention ，然后通过残差连接与源输入相加，在经过一个Intermediate层(线性层加激活函数)得到最终输出
* 由编码器层输出的向量经过prediction_heads(预测分类头，线性层+激活函数)得到最后的类别概率
* BERT的注意力机制与Transformer架构中的注意力的差异是:
* * BERT在计算完注意力分数之后，通过Position Embedding层(线性层)来融入融入相对位置信息。

## Encoder-Decoder PLM (编码器->解码器预训练语言模型)

### T5 预训练语言模型

#### 模型架构

* Tokenizer层：负责分词、编码
* EncoderLayers层：多个 多头注意力机制、前馈神经网络、归一化(Norm)层
* DncoderLayers层：多个 多头注意力机制、前馈神经网络、归一化(Norm)层，多了一个Encoder-Decoder Attention 结构，用于捕捉输入和输出序列之间的依赖关系。

#### 预训练

在输入文本中随机遮蔽15%的token，然后让模型预测这些被遮蔽的token

#### 大一统思想

* 所有的 NLP 任务都可以统一为文本到文本的任务
* 对于不同的NLP任务，每次输入前都会加上一个任务描述前缀，明确指定当前任务的类型。例如：“summarize: ”用于摘要任务，或“translate English to German: ”用于翻译任务
* 大一统的作用：简化任务处理流程、增强了模型的通用性和适应性

## Decoder-Only PLM (只有解码器的预训练语言模型)

### 代表模型：GPT 也是LLM的大门
* 先在海量的无监督预料上预训练，然后再在每个特定任务上进行微调。

#### 架构
* 在输入的向量通过Embedding层时，沿用Transformer的三角函数进行绝对位置编码，再使用12层解码器层(归一化层调到了注意力机制层前边)得到最终输出

#### 预训练-因果语言模型 CLM
* 基于自然语言序列前的所有token来预测下一个token


#### GPT模型的发展
* GPT-2 的核心改进是大幅增加了预训练数据集和模型体量。
* GPT-2 的重大突破zero-shot(零样本学习)则强调不使用任何训练样本，直接通过向预训练语言模型描述问题来去解决该问题。
* GPT-3 比GPT-2增大了模型体量跟预训练数据集，采用了稀疏注意力来取代传统的注意力机制，由zero-shot(零样本学习)变为few-shot(少样本学习)

### 智普大模型基座 GLM

## 大语言模型（LLM）

### 大语言模型的能力
* 涌现能力(Emergent Abilties)：量变引起了质变；有些能力在小模型中不明显，但是在大模型中特别突出
* 上下文的学习(In-context Learning)：无需额外的训练或参数更新，仅通过理解上下文来进行相应的输出
* 指令微调(Instruction Following)：通过自然语言描述的多任务数据进行微调，模型能够理解未见的任务指令，并且能够根据指令执行任务。例如Prompt、Agent、skills、MCP、Harnees等。
* 逐步推理(Step by Step Reasoning)。

### 训练语言模型的流程

#### 预训练 Pretrain
* 对海量无监督数据对随机初始化的模型参数进行训练，寻找最优参数集，最后让模型能根据前一个token预测下一个token
* 一开始的数据处理也很重要，主要包括：
* * 文档准备：从网站上爬取获得自然语言文档，进行URL过滤、文档提取、语言选择等操作
* * 语料过滤：去除低质量、无意义、乱码、广告等数据
* * 语料去重：去掉一些相似度非常高的语句，因为大量重复文本会影响模型的泛化能力

#### 有监督微调 SFT
* 通过多种类型跟风格的指令进行大语言模型的微调，使它更好的理解指令和回复指令
* SFT阶段跟多轮对话有关系，要使模型进行多轮对话，那么就在这个阶段构造出多轮对话的格式让模型利用之前学到的只是来进行回答
* 例如：

```json
{
    "instruction":"将下列文本翻译成英文：",
    "input":"今天天气真好",
    "output":"Today is a nice day！"
}
```

#### 人类反馈强化学习 RLHF
* 通过强化学习来让LLM与人类价值观对齐
* 所以大模型的训练过程可以分为预训练和对齐(alignment)
* * 对齐分为：
* * * 有监督微调(SFT):让大模型与人类的指令对齐
* * * 人类反馈强化学习(RLHF):让大模型与人类的价值观对齐


##### RLHF主要包括两块：奖励模型跟强化学习中的近端策略优化算法
* 奖励模型(RM):对LLM输出的每个回复进行打分(通常一个问题输出多个，然后进行排序)，根据分数来进行优化训练
* PPO:对生成的回复计算KL散度、根据KL散度和打分在输入奖励函数进行更新。

# 大模型搭建以及训练流程（因为目前没卡，写伪代码）

# transformers 预训练

## 初始化模型 Qwen/Qwen2.5-1.5B

### 下载模型

In [4]:
import os
# 设置环境变量，此处使用 HuggingFace 镜像网站
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 下载模型
os.system('huggingface-cli download --resume-download Qwen/Qwen2.5-1.5B --local-dir your_local_dir')

0

### 加载模型

In [9]:
# 加载定义好的模型参数-此处以 Qwen-2.5-1.5B 为例
# 使用 transforemrs 的 Config 类进行加载
from transformers import AutoConfig
# 下载参数的本地路径
model_path = "./your_local_dir"
config = AutoConfig.from_pretrained(model_path)
config

Qwen2Config {
  "_name_or_path": "./your_local_dir",
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 131072,
  "max_window_layers": 28,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.1",
  "use_cache": true,
  "use_mrope": false,
  "use_sliding_window": false,
  "vocab_size": 151936
}

### 基于加载好的配置对象生成对应的模型

In [11]:
# 使用该配置加载一个下载好的模型
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_config(config,trust_remote_code=True)
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

### 初始化Tokenier分词器

In [35]:
# 加载一个预训练好的tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Qwen2TokenizerFast(name_or_path='./your_local_dir', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normal

### 加载预训练数据集
* 下载数据集 序列猴子 最小的那个文件  download_dataset.sh
mobvoi_seq_monkey_classical_chs_open_corpus.tar.bz2
[序列猴子数据集](https://www.modelscope.cn/datasets/ddzhu123/seq-monkey/files)

In [40]:
# 加载预训练数据
from datasets import load_dataset

ds = load_dataset('json',data_files='autodl-tmp/dataset/pretrain_data/mobvoi_seq_monkey_general_open_corpus_small.jsonl')


In [44]:
ds["text"][2]
# column_names = list(ds["train"].features)
# column_names

KeyError: 'text'

## 对数据集进行预处理

In [38]:
# 对数据集进行 tokenize
def tokenize_function(examples):
    # 使用预先加载的 tokenizer 进行分词
    output = tokenizer([item for item in examples["id"]])
    return output

# 批量处理
tokenized_datasets = ds.map(
    tokenize_function,
    batched=True,
    num_proc=10,
    remove_columns=column_names ,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)


Running tokenizer on dataset (num_proc=10):   0%|          | 0/3606402 [00:03<?, ? examples/s]


NameError: name 'tokenizer' is not defined